### SECTION 1: IMPORTS & DATA LOADING
#### Load two data sources and merge them into one clean table:
####   SOURCE A: CSV : program-level data (employment, tuition, admission)
####   SOURCE B: XLSX : school-level data  (rent, sustainability, campus feel)
#### Then immediately filter to only the 10 universities in our study.

In [2]:
import pandas as pd
import numpy as np

In [4]:
df_csv = pd.read_csv("5301_training_dataset.csv")

# The CSV uses "Western - Main Campus" but our spreadsheet uses
# "Western University". We fix this before any merging so the
# join doesn't silently drop Western rows.
df_csv["school_name"] = df_csv["school_name"].replace({
    "Western - Main Campus": "Western University",
    "Queen's University":    "Queen's University",   # already matches
})

# We drop all other Ontario universities now so they never
# appear in recommendations or distort any normalization later.
OUR_10 = [
    "University of Ottawa",
    "University of Waterloo",
    "University of Toronto",
    "McMaster University",
    "Queen's University",
    "Western University",
    "York University",
    "Carleton University",
    "University of Guelph",
    "Ontario Tech University",
]
df_csv = df_csv[df_csv["school_name"].isin(OUR_10)].copy().reset_index(drop=True)

print(f"CSV rows after filter: {len(df_csv)}")
print(df_csv["school_name"].value_counts())


CSV rows after filter: 1221
school_name
University of Toronto      163
Western University         152
University of Ottawa       139
University of Waterloo     130
McMaster University        118
University of Guelph       117
Queen's University         114
York University            106
Carleton University         97
Ontario Tech University     85
Name: count, dtype: int64


In [9]:
# Load the Excel audit sheet (school-level data) 
#Each row = one school. It has cost of living, sustainability scores,
# campus feel, equity scores, and co-op counts — things that
# apply to the whole school, not individual programs.
df_school = pd.read_excel(
    "GNG5301_Data_Audit_Spreadsheet_updated.xlsx",
    sheet_name="ML Feature Matrix",
    header=2          # row index 2 is the actual header 
)

# Keep only the columns we actually use in scoring
SCHOOL_COLS = [
    "university_name",
    "city",
    "region",
    "campus_type",          # urban / suburban / mixed
    "col_score",            # composite cost of living score (1–10)
    "sustainability_score", # 1–5 rating we assigned
    "campus_feel_score",    # 1 (large urban) → 5 (small intimate)
    "equity_score",         # 1–5 rating for EDI initiatives
    "coop_placement_rate",  # % of co-op students who got placed
    "programs_coop_count",  # number of co-op programs offered
    "intl_student_pct",     # % of student body that is international
    "residence_capacity",   # on-campus beds
    "bilingual",            # True/False
    "avg_entering_grade",   # average admitted student grade
    "qsranking_metric",     # QS rank number (lower = better)
    "total_enrollment",     # total student headcount
]
df_school = df_school[SCHOOL_COLS].copy()
df_school = df_school[df_school["university_name"].notna()].reset_index(drop=True)

print(f"\nSchool sheet rows: {len(df_school)}")
print(df_school["university_name"].tolist())



School sheet rows: 10
['University of Ottawa', 'University of Waterloo', 'University of Toronto', 'McMaster University', "Queen's University", 'Western University', 'York University', 'Carleton University', 'University of Guelph', 'Ontario Tech University']


In [10]:
# Merge the two sources
# We join on school name. Every program row in the CSV gets
# the school-level attributes attached to it from the spreadsheet.
# This is a LEFT JOIN — every CSV row is kept, and school
# attributes are added where the name matches.
df = df_csv.merge(
    df_school,
    left_on="school_name",
    right_on="university_name",
    how="left"
)

# Sanity check: every row should have a matched school
unmatched = df[df["col_score"].isna()]["school_name"].unique()
if len(unmatched) > 0:
    print(f"\n⚠ WARNING — unmatched schools (no spreadsheet data): {unmatched}")
else:
    print(f"\n✅ All {len(df)} rows matched successfully.")
    print(f"   Columns available: {len(df.columns)}")


✅ All 1221 rows matched successfully.
   Columns available: 38


### SECTION 2 — FEATURE ENGINEERING & NORMALIZATION
Raw numbers (tuition in dollars, QS rank in integers, employment as percentages) can't be compared directly. This section transforms everything into 0–1 scores where:
-  1.0 = best possible outcome for the student
- 0.0 = worst possible outcome for the student
#### Also:
- Map 50+ program names into 11 broad categories
- Keep only the most recent year per school-program pair
- Build a data quality score to flag sparse/unreliable rows

In [11]:
# Normalization helper 
# # minmax scales a column so its minimum becomes 0 and its
# maximum becomes 1. We normalize WITHIN our 10-uni
# dataset so comparisons are relative to each other, not
# to all Ontario universities.
#
# The edge case guard (mx == mn) handles columns where every
# value is identical — without it, we'd get division by zero.
# In that case we return 0.5 (neutral) for all rows.

def minmax(series):
    s  = pd.to_numeric(series, errors="coerce")
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0.5, index=s.index)
    return (s - mn) / (mx - mn)


In [12]:

# Program category mapping 
# The CSV has 50+ specific program names like
# "Kinesiology, Recreation & Phys. Educ." which are too
# granular for a student quiz. We group them into 11 broad
# categories that match the quiz options we'll build later.
#
# Any program not in the mapping falls into "Other" — we
# print those at the end so they can be reviewed.

CATEGORY_MAP = {
    # Engineering & Technology
    "Engineering":                              "Engineering & Technology",
    "Computer Science":                         "Engineering & Technology",
    "Mathematics":                              "Engineering & Technology",
    "Physical Science":                         "Engineering & Technology",
    # Health & Medical
    "Nursing":                                  "Health & Medical",
    "Pharmacy":                                 "Health & Medical",
    "Dentistry":                                "Health & Medical",
    "Optometry":                                "Health & Medical",
    "Medicine & Related Programs":              "Health & Medical",
    "Other Health Professions":                 "Health & Medical",
    "Therapy & Rehabilitation":                 "Health & Medical",
    "Kinesiology, Recreation & Phys. Educ.":   "Health & Medical",
    "Veterinary Medicine":                      "Health & Medical",
    # Science & Environment
    "Agriculture & Biological Science":         "Science & Environment",
    "Forestry":                                 "Science & Environment",
    # Food Science
    "Food Science & Nutrition":                 "Food Science",
    # Architecture
    "Architecture":                             "Architecture",
    # Business
    "Business & Commerce":                      "Business",
    # Arts, Humanities & Social Sciences
    "Fine & Applied Arts":                      "Arts, Humanities & Social Sciences",
    "Humanities":                               "Arts, Humanities & Social Sciences",
    "Journalism":                               "Arts, Humanities & Social Sciences",
    "Social Science":                           "Arts, Humanities & Social Sciences",
    "Other Arts & Science":                     "Arts, Humanities & Social Sciences",
    "Theology":                                 "Arts, Humanities & Social Sciences",
    # Education
    "Education":                                "Education",
    # Law
    "Law":                                      "Law",
}

def classify_category(program_name):
    if pd.isna(program_name):
        return "Other"
    return CATEGORY_MAP.get(str(program_name).strip(), "Other")

df["category"] = df["program_name"].apply(classify_category)

# Flag any programs that fell through to "Other" so we can
# review and add them to the map if needed
unmapped = df[df["category"] == "Other"]["program_name"].unique()
if len(unmapped) > 0:
    print(f"⚠  {len(unmapped)} unmapped program(s) → 'Other': {list(unmapped)}")
else:
    print("✅ All programs mapped to a category.")

print("\nCategory distribution:")
print(df["category"].value_counts())


✅ All programs mapped to a category.

Category distribution:
category
Arts, Humanities & Social Sciences    333
Engineering & Technology              306
Health & Medical                      264
Science & Environment                  82
Business                               80
Law                                    48
Education                              48
Architecture                           32
Food Science                           28
Name: count, dtype: int64


In [14]:
# Keep only the most recent year per school-program 
# The CSV has data from 2014–2022 for each program. We only
# want the most recent snapshot — using old tuition or
# employment data from 2014 would give stale recommendations.
#
# groupby + tail(1) after sorting by year gives us the last
# (most recent) row for each school-program combination.

latest = (
    df.sort_values(["school_name", "program_name", "year"])
      .groupby(["school_name", "program_name"], as_index=False)
      .tail(1)
      .copy()
      .reset_index(drop=True)
)

print(f"\n✅ Reduced to {len(latest)} rows (one per school-program, most recent year)")


✅ Reduced to 163 rows (one per school-program, most recent year)


In [15]:
#  Build scored dimensions
# For each dimension we want to score, we apply minmax and
# flip where needed (higher raw value = worse for student).
#
# FLIP LOGIC: cost and QS rank are inverted because lower is
# better. We use (1 - minmax) to flip so that the cheapest
# school scores 1.0 and the most expensive scores 0.0.

# Employment — higher % employed = better → no flip needed
latest["employment_score"] = minmax(latest["employment_6m"])

# Tuition cost — cheaper = better → flip with (1 - minmax)
latest["domestic_cost_score"]  = 1 - minmax(latest["domestic_total_cost"])
latest["intl_cost_score"]      = 1 - minmax(latest["international_total_cost"])

# Cost of living (from spreadsheet) — lower col_score = cheaper = better → flip
latest["col_score_norm"] = 1 - minmax(latest["col_score"])

# QS ranking — lower rank number = better university → flip
# Universities with no QS rank (NaN) get penalized to 25%
# above the worst rank in the dataset rather than treated
# as missing, so they don't accidentally appear top-ranked.
qs_numeric = pd.to_numeric(latest["qsranking_metric"], errors="coerce")
worst_rank  = qs_numeric.dropna().max()
qs_filled   = qs_numeric.fillna(worst_rank * 1.25)
latest["ranking_score"] = 1 - minmax(qs_filled)

# Sustainability score (1–5 from spreadsheet) — higher = better → no flip
latest["sustainability_score_norm"] = minmax(latest["sustainability_score"])

# Equity/diversity score (1–5 from spreadsheet) — higher = better → no flip
latest["diversity_score_norm"] = minmax(latest["equity_score"])

# Campus feel (1–5 from spreadsheet) — this one is NOT
# normalized the same way. A student will specify their
# preference (urban vs intimate), so we keep the raw 1–5
# value and compare it to the student's preference at
# scoring time rather than normalizing it now.
# (We'll handle this in Section 3.)

# Co-op placement rate — higher = better → no flip
# Fill missing with the dataset median so schools without
# published rates aren't penalized to 0.
latest["coop_rate_score"] = minmax(
    latest["coop_placement_rate"].fillna(
        latest["coop_placement_rate"].median()
    )
)

print("\n✅ Scored dimensions built:")
score_cols = [
    "employment_score", "domestic_cost_score", "intl_cost_score",
    "col_score_norm", "ranking_score", "sustainability_score_norm",
    "diversity_score_norm", "coop_rate_score"
]
print(latest[["school_name"] + score_cols].groupby("school_name").mean().round(3).to_string())


✅ Scored dimensions built:
                         employment_score  domestic_cost_score  intl_cost_score  col_score_norm  ranking_score  sustainability_score_norm  diversity_score_norm  coop_rate_score
school_name                                                                                                                                                                     
Carleton University                 0.711                1.000            1.000           0.425          0.200                      0.333                   0.5            0.424
McMaster University                 0.764                0.994            0.785           0.644          0.853                      0.333                   0.5            0.254
Ontario Tech University             0.709                0.221            0.700           0.954          0.000                      0.000                   0.0            0.254
Queen's University                  0.894                0.000            0.108        

In [16]:
# Data quality score 
# # Not all rows are equally trustworthy. A row that has:
#   - admission data     (has_any_admission_feature)
#   - school-level data  (has_any_school_level_feature)
#   - a known QS rank    (qsranking_metric is not NaN)
# is more reliable than one missing all three.
#
# This 0–1 score is later used as a confidence multiplier
# so sparse rows can't confidently top the rankings.
# Weights reflect importance: admission data matters most
# because it directly affects student eligibility.

admission_flag  = pd.to_numeric(latest["has_any_admission_feature"],    errors="coerce").fillna(0)
school_flag     = pd.to_numeric(latest["has_any_school_level_feature"],  errors="coerce").fillna(0)
rank_known_flag = qs_numeric.notna().astype(float)

latest["data_quality_score"] = (
    0.45 * admission_flag +
    0.35 * school_flag   +
    0.20 * rank_known_flag
)

print("\n✅ Data quality score distribution:")
print(latest["data_quality_score"].describe().round(3))


✅ Data quality score distribution:
count    163.000
mean       0.847
std        0.210
min        0.350
25%        0.550
50%        1.000
75%        1.000
max        1.000
Name: data_quality_score, dtype: float64


In [17]:
# Category breadth score
# If a student selects "Engineering," a school offering 8
# Engineering programs is more likely to have a good fit
# than one offering only 1. We give a small breadth bonus
# to schools with more options in the student's category.
#
# This is computed per school-category pair, then merged back
# onto the main table.

category_breadth = (
    latest.groupby(["school_name", "category"])["program_name"]
          .nunique()
          .reset_index(name="category_program_count")
)
latest = latest.merge(category_breadth, on=["school_name", "category"], how="left")
latest["breadth_score"] = minmax(latest["category_program_count"]).fillna(0.0)

print("\n✅ Feature engineering complete.")
print(f"   Final table: {len(latest)} rows × {len(latest.columns)} columns")
print(f"   Schools: {latest['school_name'].nunique()}")
print(f"   Unique programs: {latest['program_name'].nunique()}")
print(f"   Categories: {sorted(latest['category'].unique())}")


✅ Feature engineering complete.
   Final table: 163 rows × 50 columns
   Schools: 10
   Unique programs: 26
   Categories: ['Architecture', 'Arts, Humanities & Social Sciences', 'Business', 'Education', 'Engineering & Technology', 'Food Science', 'Health & Medical', 'Law', 'Science & Environment']


## Section 3 — Student Quiz

Collects the student's preferences through a short guided questionnaire and returns a structured dictionary that Section 4 uses directly to weight and rank universities.

**Design principles:**
- Questions are grouped into 3 logical blocks so the student isn't overwhelmed
- Every input is validated — invalid entries re-prompt rather than crashing
- A summary is shown at the end so the student can confirm before recommendations are generated
- All importance ratings use a 1–5 scale: 1 = not important at all, 3 = somewhat important, 5 = essential / top priority

In [18]:
#Helper functions for asking quiz questions

def ask_choice(prompt, options):
    """
    Shows a numbered list of options and returns the chosen one.
    Keeps re-prompting until the student enters a valid number.
    This replaces the original ask_selection() but is cleaner
    because it shows the number range in the prompt itself.
    """
    print(f"\n{prompt}")
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    while True:
        raw = input(f"  → Enter a number (1–{len(options)}): ").strip()
        if raw.isdigit() and 1 <= int(raw) <= len(options):
            return options[int(raw) - 1]
        print(f"  ⚠  Please enter a number between 1 and {len(options)}.")


def ask_importance(question):
    """
    Asks a 1–5 importance rating with labels at each anchor
    so the student understands what the numbers mean.
    Returns an integer 1–5.
    """
    print(f"\n{question}")
    print("  1 = Not important at all")
    print("  2 = Slightly important")
    print("  3 = Somewhat important")
    print("  4 = Important")
    print("  5 = Essential — top priority")
    while True:
        raw = input("  → Your rating (1–5): ").strip()
        if raw.isdigit() and 1 <= int(raw) <= 5:
            return int(raw)
        print("  ⚠  Please enter a number between 1 and 5.")


def ask_float(question, lo, hi, default):
    """
    Asks for a numeric value within a range.
    If the student just presses Enter, the default is used.
    Used for academic average where a precise number matters.
    """
    print(f"\n{question} [{lo}–{hi}, default: {default}]")
    while True:
        raw = input("  → ").strip()
        if raw == "":
            return default
        try:
            val = float(raw)
            if lo <= val <= hi:
                return val
        except ValueError:
            pass
        print(f"  ⚠  Please enter a number between {lo} and {hi}.")

In [19]:
# Available categories
# Must match exactly what Section 2 produced — these are the
# categories that actually exist in our filtered dataset.
# "No preference" means the student doesn't want to filter
# by category and all programs are considered.

CATEGORIES = [
    "No preference",
    "Engineering & Technology",
    "Health & Medical",
    "Business",
    "Science & Environment",
    "Arts, Humanities & Social Sciences",
    "Law",
    "Education",
    "Architecture",
    "Food Science",
]

# Campus feel options — these map to the 1–5 scale in our
# spreadsheet (1 = large urban, 5 = small intimate).
# We store the numeric value alongside the label so Section 4
# can compute the distance between the student's preference
# and each school's campus_feel_score.
CAMPUS_FEEL_OPTIONS = {
    "Large urban research university (like U of T or York)":    1,
    "Large but campus-focused (like Waterloo or Western)":      3,
    "Medium-sized with strong community feel (like Queen's)":   3,
    "Smaller and more intimate (like Guelph or Ontario Tech)":  4,
    "No preference":                                            None,
}

In [20]:
#The Quiz function
def run_quiz():
    """
    Runs the full questionnaire and returns a dictionary with
    all student preferences. This dict is the only thing
    Section 4 needs — the quiz and the scorer are fully decoupled.
    """

    print("\n" + "="*56)
    print("  Ontario University Recommender — Student Profile")
    print("="*56)
    print("Answer a few short questions and we'll find your")
    print("best-fit universities from Ontario's top schools.")
    print("(Press Enter to accept the default where shown.)")

    # ── Block 1: Who are you? ────────────────────────────────
    print("\n── BLOCK 1 of 3: About You ─────────────────────────")

    student_status = ask_choice(
        "Are you a domestic or international student?",
        ["Domestic", "International"]
    )

    category = ask_choice(
        "Which academic area are you most interested in?",
        CATEGORIES
    )

    avg = ask_float(
        "What is your current academic average (%)?\n"
        "  This helps us assess your admission fit for each program.",
        lo=60.0, hi=100.0, default=80.0
    )

    # ── Block 2: What matters to you financially? ────────────
    print("\n── BLOCK 2 of 3: Your Priorities ───────────────────")
    print("Rate each factor from 1 (not important) to 5 (essential).\n")

    imp_jobs = ask_importance(
        "💼  Finding a job quickly after graduation"
    )
    imp_cost = ask_importance(
        "💰  Keeping your total costs low\n"
        "    (tuition + cost of living)"
    )
    imp_rank = ask_importance(
        "🏅  School prestige and global ranking"
    )
    imp_sust = ask_importance(
        "🌱  Sustainability and environmental initiatives on campus"
    )
    imp_div  = ask_importance(
        "🤝  Diversity, equity, and inclusion programs"
    )
    imp_coop = ask_importance(
        "🔧  Co-op / work-integrated learning availability"
    )

    # ── Block 3: Campus feel ─────────────────────────────────
    print("\n── BLOCK 3 of 3: Campus Environment ────────────────")

    feel_label = ask_choice(
        "What kind of campus environment do you prefer?",
        list(CAMPUS_FEEL_OPTIONS.keys())
    )
    feel_value = CAMPUS_FEEL_OPTIONS[feel_label]

    # ── Confirmation summary ─────────────────────────────────
    # Show everything back to the student before scoring.
    # This gives them a chance to re-run if they made a mistake,
    # and it makes the system feel transparent.

    print("\n" + "="*56)
    print("  Your Profile — Please Confirm")
    print("="*56)
    print(f"  Student type:     {student_status}")
    print(f"  Area of interest: {category}")
    print(f"  Academic average: {avg:.1f}%")
    print(f"  Campus feel:      {feel_label}")
    print()
    print(f"  Priority ratings (1 = low, 5 = essential):")
    print(f"    Employment after graduation : {imp_jobs}/5")
    print(f"    Low cost                    : {imp_cost}/5")
    print(f"    School ranking / prestige   : {imp_rank}/5")
    print(f"    Sustainability              : {imp_sust}/5")
    print(f"    Diversity & equity          : {imp_div}/5")
    print(f"    Co-op availability          : {imp_coop}/5")
    print("="*56)

    confirm = input("\nLooks good? Press Enter to get recommendations, "
                    "or type 'r' to restart: ").strip().lower()
    if confirm == "r":
        print("\nRestarting quiz...\n")
        return run_quiz()   # recursive restart — clean and simple

    # ── Return structured dict ───────────────────────────────
    # Everything downstream uses this dictionary.
    # Section 4 only reads from here — it never touches
    # raw quiz inputs directly. This makes the quiz and
    # the scorer independently testable and modifiable.

    return {
        "student_status": student_status,   # "Domestic" or "International"
        "category":       category,         # one of CATEGORIES
        "avg":            avg,              # float 60–100
        "feel_pref":      feel_value,       # int 1–4 or None
        "imp_jobs":       imp_jobs,         # int 1–5
        "imp_cost":       imp_cost,         # int 1–5
        "imp_rank":       imp_rank,         # int 1–5
        "imp_sust":       imp_sust,         # int 1–5
        "imp_div":        imp_div,          # int 1–5
        "imp_coop":       imp_coop,         # int 1–5
    }


In [21]:
student = run_quiz()
print(f"\n✅ Profile captured. Ready for scoring.")
print(f"   Raw preferences: {student}")


  Ontario University Recommender — Student Profile
Answer a few short questions and we'll find your
best-fit universities from Ontario's top schools.
(Press Enter to accept the default where shown.)

── BLOCK 1 of 3: About You ─────────────────────────

Are you a domestic or international student?
  1. Domestic
  2. International

Which academic area are you most interested in?
  1. No preference
  2. Engineering & Technology
  3. Health & Medical
  4. Business
  5. Science & Environment
  6. Arts, Humanities & Social Sciences
  7. Law
  8. Education
  9. Architecture
  10. Food Science

What is your current academic average (%)?
  This helps us assess your admission fit for each program. [60.0–100.0, default: 80.0]

── BLOCK 2 of 3: Your Priorities ───────────────────
Rate each factor from 1 (not important) to 5 (essential).


💼  Finding a job quickly after graduation
  1 = Not important at all
  2 = Slightly important
  3 = Somewhat important
  4 = Important
  5 = Essential — top pr

## Section 4 — Weighted Scoring Engine

Takes the student profile from Section 3 and the prepared program table from Section 2, and produces a ranked DataFrame of the best-matching university-program pairs.

**Pipeline:**
1. Filter candidates by category + admission eligibility
2. Convert 1–5 importance ratings → normalized weights
3. Score each candidate on 8 dimensions
4. Apply confidence multiplier (data quality + breadth)
5. Sort and return top N results

In [22]:
#Importance → weight conversion
# A student's 1–5 rating needs to become a weight (0–1) that
# reflects how much that dimension should influence the score.
#
# We use a floor + range structure:
#   weight = floor + range * scale(importance)
#   scale converts 1→0, 3→0.5, 5→1.0 linearly
#
# The floor ensures no dimension is ever completely ignored
# (even if rated 1, employment still gets some weight).
# The range controls how much the student can amplify it.
#
# After all weights are computed, we normalize by dividing
# by their sum — so they always total exactly 1.0 regardless
# of what the student entered.

def scale(importance):
    """Maps 1–5 rating linearly to 0.0–1.0."""
    return (importance - 1) / 4


def build_weights(s):
    """
    Takes the student dict and returns a normalized weight
    dict for all 8 scoring dimensions.

    Floor and range values are set so that:
      - A dimension rated 1/5 gets its floor weight only
      - A dimension rated 5/5 gets floor + range
      - Rankings and employment have higher ranges because
        they're the most differentiating factors across
        our 10 universities
      - Admission fit is fixed at 0.15 — it always matters
        regardless of student preferences
      - Data quality is fixed at 0.05 — a background penalty
        that the student never controls
    """
    raw = {
        "employment":    0.05 + 0.25 * scale(s["imp_jobs"]),
        "cost":          0.05 + 0.20 * scale(s["imp_cost"]),
        "ranking":       0.03 + 0.18 * scale(s["imp_rank"]),
        "sustainability":0.02 + 0.10 * scale(s["imp_sust"]),
        "diversity":     0.02 + 0.10 * scale(s["imp_div"]),
        "coop":          0.03 + 0.15 * scale(s["imp_coop"]),
        "admission":     0.15,   # fixed — always relevant
        "data_quality":  0.05,   # fixed — background penalty
    }
    total = sum(raw.values())
    return {k: v / total for k, v in raw.items()}


In [23]:
# Admission fit 
# Rather than a binary pass/fail, we score how well the
# student's average aligns with the program's admission
# average. The graduated scale rewards students who are
# comfortably above the cutoff and still keeps borderline
# candidates in the running with a lower score.
#
# The hard filter (cutoff=-5) removes programs where the
# student is more than 5 points below the average — those
# are genuine reaches that shouldn't appear in top-3.

def admission_fit_score(student_avg, program_avg):
    """Returns a 0.0–1.0 fit score based on grade difference."""
    if pd.isna(program_avg):
        return 0.45          # neutral if program has no data
    diff = float(student_avg) - float(program_avg)
    if diff >=  5:  return 1.00  # well above — strong match
    if diff >=  2:  return 0.85  # comfortably above
    if diff >=  0:  return 0.70  # right at the cutoff
    if diff >= -2:  return 0.50  # slightly below
    if diff >= -4:  return 0.35  # notably below
    return 0.20                  # well below — poor fit


def passes_admission_filter(student_avg, program_avg, cutoff=-5):
    """Hard filter — returns False if student is too far below cutoff."""
    if pd.isna(program_avg):
        return True             # keep if no data
    return (float(student_avg) - float(program_avg)) >= cutoff

In [24]:
#Campus feel matching ─────────────────────────────────
# Campus feel is different from the other dimensions because
# it's a preference match, not a "higher is better" score.
# A student who wants a small campus (preference=4) should
# score Guelph (feel=4) highly and uToronto (feel=1) low.
#
# We compute the absolute distance between preference and
# actual feel, then convert to a 0–1 score where 0 distance
# = 1.0 and maximum distance (4 points) = 0.0.
# If the student chose "No preference", everyone gets 0.5.

def campus_feel_match(student_pref, school_feel):
    """Returns 0.0–1.0 campus feel match score."""
    if student_pref is None:
        return 0.5                       # no preference → neutral
    if pd.isna(school_feel):
        return 0.5                       # no data → neutral
    distance = abs(float(student_pref) - float(school_feel))
    return 1.0 - (distance / 4.0)       # max distance is 4 (1 vs 5)

In [25]:
# Select cost score based on student type 
# We built both domestic and international cost scores in
# Section 2. Here we pick the right one based on the
# student's status from the quiz.

def get_cost_score(row, student_status):
    if student_status == "Domestic":
        return row["domestic_cost_score"]
    return row["intl_cost_score"]

In [26]:
# Main scoring function 

def score_candidates(latest, student, top_n=3):
    """
    Filters and scores all program-school rows against the
    student profile. Returns the top_n results as a DataFrame.
    """

    # Step 1 — Category filter
    # If the student chose a specific category, we only
    # consider programs in that category. "No preference"
    # means all programs remain in the candidate pool.
    if student["category"] != "No preference":
        cand = latest[latest["category"] == student["category"]].copy()
    else:
        cand = latest.copy()

    if cand.empty:
        print(f"⚠  No programs found for category: {student['category']}")
        return pd.DataFrame()

    cand = cand.reset_index(drop=True)

    # Step 2 — Admission filter
    # Remove programs where the student is more than 5 points
    # below the admission average. These are unrealistic matches
    # that shouldn't appear in recommendations.
    before = len(cand)
    cand = cand[
        cand["admission_overall_average"].apply(
            lambda avg: passes_admission_filter(student["avg"], avg)
        )
    ].copy().reset_index(drop=True)
    removed = before - len(cand)
    if removed > 0:
        print(f"   ℹ  {removed} program(s) removed by admission filter.")

    if cand.empty:
        print("⚠  No programs remain after admission filter.")
        print("   Try a lower academic average or 'No preference' for category.")
        return pd.DataFrame()

    # Step 3 — Build weights from student's importance ratings
    weights = build_weights(student)

    # Step 4 — Compute each scoring component
    # Each component = weight × normalized score (0–1)
    # This makes components directly comparable and additive.

    cand["adm_score"]   = cand["admission_overall_average"].apply(
                              lambda a: admission_fit_score(student["avg"], a))

    cand["cost_score"]  = cand.apply(
                              lambda r: get_cost_score(r, student["student_status"]),
                              axis=1).fillna(0.5)

    cand["feel_score"]  = cand["campus_feel_score"].apply(
                              lambda f: campus_feel_match(student["feel_pref"], f))

    # Combined cost: 70% tuition cost + 30% cost of living
    # This reflects that tuition is the bigger financial
    # burden but COL meaningfully affects total expenditure.
    cand["combined_cost"] = (
        0.70 * cand["cost_score"] +
        0.30 * cand["col_score_norm"].fillna(0.5)
    )

    # Weighted components
    cand["c_employment"]    = weights["employment"]    * cand["employment_score"].fillna(0.5)
    cand["c_cost"]          = weights["cost"]          * cand["combined_cost"]
    cand["c_ranking"]       = weights["ranking"]       * cand["ranking_score"].fillna(0.1)
    cand["c_sustainability"]= weights["sustainability"]* cand["sustainability_score_norm"].fillna(0.5)
    cand["c_diversity"]     = weights["diversity"]     * cand["diversity_score_norm"].fillna(0.5)
    cand["c_coop"]          = weights["coop"]          * cand["coop_rate_score"].fillna(0.5)
    cand["c_admission"]     = weights["admission"]     * cand["adm_score"]
    cand["c_feel"]          = 0.08                     * cand["feel_score"]

    # Raw score = sum of all weighted components
    cand["raw_score"] = (
        cand["c_employment"]     +
        cand["c_cost"]           +
        cand["c_ranking"]        +
        cand["c_sustainability"] +
        cand["c_diversity"]      +
        cand["c_coop"]           +
        cand["c_admission"]      +
        cand["c_feel"]
    )

    # Step 5 — Confidence multiplier
    # Penalizes rows with sparse or low-quality data so they
    # can't top the rankings purely by luck of missing values.
    # Range: 0.80 (no data) → 1.00 (full data + broad category)
    cand["confidence"] = (
        0.80
        + 0.12 * cand["data_quality_score"].fillna(0.0)
        + 0.08 * cand["breadth_score"].fillna(0.0)
    )

    cand["final_score"] = cand["raw_score"] * cand["confidence"]

    # Step 6 — Sort and return top N
    # Primary sort: final_score descending
    # Tiebreakers: employment rate, then ranking score
    cand = cand.sort_values(
        ["final_score", "employment_score", "ranking_score"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return cand.head(top_n)


In [27]:
# Run the scorer 
print("Scoring candidates against your profile...\n")
results = score_candidates(latest, student, top_n=3)

if not results.empty:
    print(f"✅ Top {len(results)} matches found.")
    print(results[[
        "school_name", "program_name", "category",
        "final_score", "c_employment", "c_cost",
        "c_ranking", "c_admission"
    ]].round(3).to_string(index=False))

Scoring candidates against your profile...

   ℹ  21 program(s) removed by admission filter.
✅ Top 3 matches found.
         school_name     program_name                 category  final_score  c_employment  c_cost  c_ranking  c_admission
University of Ottawa      Mathematics Engineering & Technology        0.722         0.287   0.152      0.065        0.050
 Carleton University Physical Science Engineering & Technology        0.669         0.225   0.198      0.014        0.122
University of Guelph Physical Science Engineering & Technology        0.656         0.240   0.217      0.035        0.029


## Section 5 — Output Display

Transforms the ranked results DataFrame into readable recommendation cards a student can actually understand.

**Each recommendation shows:**
- School name, program, and category
- Key facts: cost, admission fit, employment rate, ranking
- A plain-English "why this matched you" explanation
- A visual score bar for each priority the student rated

Followed by a side-by-side comparison table of all 3.

In [28]:
# Formatting helpers 
def score_bar(score, width=10):
    """
    Converts a 0–1 score into a visual bar of filled/empty
    blocks so the student can see relative performance at
    a glance without needing to interpret decimals.

    Example: score=0.7, width=10 → '███████░░░'
    """
    filled = round(score * width)
    return "█" * filled + "░" * (width - filled)


def admission_label(student_avg, program_avg):
    """
    Returns a plain-English admission fit label based on
    the difference between the student's average and the
    program's admission average.
    """
    if pd.isna(program_avg):
        return "No cutoff data"
    diff = float(student_avg) - float(program_avg)
    if diff >=  5:  return "✅ Strong fit"
    if diff >=  2:  return "✅ Good fit"
    if diff >=  0:  return "⚠️  At the cutoff"
    if diff >= -3:  return "⚠️  Slight reach"
    return                 "❌ Reach"


def format_cost(row, student_status):
    """Returns the right annual cost figure with $ formatting."""
    if student_status == "Domestic":
        val = row.get("domestic_total_cost", None)
    else:
        val = row.get("international_total_cost", None)
    if pd.isna(val) or val is None:
        return "N/A"
    return f"${val:,.0f}/yr"


def rank_display(row):
    """Returns a readable QS rank string."""
    raw = row.get("qs_wur_2026_rank", None)
    if pd.isna(raw) or raw is None:
        return "Not ranked"
    return str(raw)


In [29]:
#Why-matched explanation
# Identifies the top 2 scoring components for this result
# and returns a plain-English sentence.
# We only mention dimensions the student rated ≥ 3 so the
# explanation reflects their stated priorities, not the
# system's internal mechanics.

COMPONENT_LABELS = {
    "c_employment":     "strong post-graduation employment rates",
    "c_cost":           "affordable tuition and cost of living",
    "c_ranking":        "strong global ranking and reputation",
    "c_sustainability": "sustainability initiatives on campus",
    "c_diversity":      "diversity and equity programs",
    "c_coop":           "co-op and work-integrated learning",
    "c_admission":      "a strong academic profile match",
}

IMPORTANCE_KEYS = {
    "c_employment":     "imp_jobs",
    "c_cost":           "imp_cost",
    "c_ranking":        "imp_rank",
    "c_sustainability": "imp_sust",
    "c_diversity":      "imp_div",
    "c_coop":           "imp_coop",
    "c_admission":      None,        # always included — no importance key
}

def why_matched(row, student):
    """
    Returns a one-sentence plain-English explanation of
    the top reasons this university matched the student.
    """
    scored = {}
    for col, label in COMPONENT_LABELS.items():
        imp_key = IMPORTANCE_KEYS[col]
        # Always include admission; for others, only include
        # if student rated the dimension ≥ 3 (meaningful priority)
        if imp_key is None or student.get(imp_key, 0) >= 3:
            scored[label] = row.get(col, 0)

    if not scored:
        return "This program aligns well with your overall profile."

    # Pick the top 2 reasons by component score
    top = sorted(scored, key=scored.get, reverse=True)[:2]

    if len(top) == 1:
        return f"This match is driven by {top[0]}."
    return f"This match is driven by {top[0]} and {top[1]}."

In [30]:
#Priority score bars
# Shows a visual bar for each dimension the student rated ≥ 3.
# This makes it easy to see at a glance how this school
# performs on what the student actually cares about.

PRIORITY_DISPLAY = [
    ("imp_jobs", "c_employment",     "💼 Employment"),
    ("imp_cost", "c_cost",           "💰 Cost"),
    ("imp_rank", "c_ranking",        "🏅 Ranking"),
    ("imp_sust", "c_sustainability", "🌱 Sustainability"),
    ("imp_div",  "c_diversity",      "🤝 Diversity"),
    ("imp_coop", "c_coop",           "🔧 Co-op"),
]

def priority_bars(row, student, weights):
    """
    Prints a bar for each dimension the student rated ≥ 3,
    showing how this school performs on that dimension.
    The bar shows the raw normalized score (0–1) for that
    dimension, not the weighted component — so it reflects
    the school's actual performance, not its weight.
    """
    # Map component cols back to their raw score cols
    raw_score_map = {
        "c_employment":     "employment_score",
        "c_cost":           "combined_cost",
        "c_ranking":        "ranking_score",
        "c_sustainability": "sustainability_score_norm",
        "c_diversity":      "diversity_score_norm",
        "c_coop":           "coop_rate_score",
    }
    lines = []
    for imp_key, comp_col, label in PRIORITY_DISPLAY:
        if student.get(imp_key, 0) >= 3:
            raw_col  = raw_score_map[comp_col]
            score    = float(row.get(raw_col, 0.5))
            bar      = score_bar(score)
            pct      = f"{score*100:.0f}%"
            lines.append(f"    {label:<20} {bar}  {pct}")
    return "\n".join(lines)

In [31]:
# Recommendation card

def print_card(rank, row, student, weights):
    """
    Prints a single formatted recommendation card.
    """
    school   = row["school_name"]
    program  = row["program_name"]
    category = row["category"]
    emp      = row.get("employment_6m", None)
    emp_str  = f"{emp:.1f}%" if pd.notna(emp) else "N/A"
    adm      = row.get("admission_overall_average", None)
    adm_str  = f"{adm:.1f}%" if pd.notna(adm) else "N/A"
    cost_str = format_cost(row, student["student_status"])
    qs_str   = rank_display(row)
    fit_str  = admission_label(student["avg"], adm)
    why      = why_matched(row, student)
    bars     = priority_bars(row, student, weights)

    medal = ["🥇", "🥈", "🥉"][rank]

    print(f"""
{'='*58}
  {medal}  MATCH #{rank+1}
{'='*58}
  🎓  {school}
      {program}  |  {category}

  📋  Key Facts
  ─────────────────────────────────────────────────
  💵  Annual cost ({student['student_status'].lower()}):  {cost_str}
  📊  Admission average:           {adm_str}  →  {fit_str}
  💼  6-month employment rate:     {emp_str}
  🌍  QS World Rank:               {qs_str}

  ✨  Why this matched you
  ─────────────────────────────────────────────────
  {why}

  📈  Your priority scores
  ─────────────────────────────────────────────────
{bars}
{'='*58}""")


In [32]:
# Comparison table 
# After the cards, show a compact side-by-side table so the
# student can compare the top 3 on the same dimensions at once.

def print_comparison(results, student):
    """
    Prints a side-by-side comparison table for all results.
    """
    print("\n" + "─"*58)
    print("  📊  SIDE-BY-SIDE COMPARISON")
    print("─"*58)

    # Header row — truncate school names to fit
    names = [r["school_name"].replace("University", "U.")
             for _, r in results.iterrows()]
    col_w = 17
    header = "  " + "".join(f"{n[:col_w]:<{col_w}}" for n in names)
    print(header)
    print("  " + "─" * (col_w * len(names)))

    # Rows
    def cmp_row(label, values):
        row_str = "  " + "".join(f"{str(v)[:col_w]:<{col_w}}" for v in values)
        print(f"  {label}")
        print(row_str)

    cmp_row("💵 Annual cost",
            [format_cost(r, student["student_status"])
             for _, r in results.iterrows()])

    cmp_row("📊 Admission avg",
            [f"{r['admission_overall_average']:.1f}%"
             if pd.notna(r["admission_overall_average"]) else "N/A"
             for _, r in results.iterrows()])

    cmp_row("💼 Employment 6m",
            [f"{r['employment_6m']:.1f}%"
             if pd.notna(r["employment_6m"]) else "N/A"
             for _, r in results.iterrows()])

    cmp_row("🌍 QS Rank",
            [rank_display(r) for _, r in results.iterrows()])

    cmp_row("🌱 Sustainability",
            [f"{int(r['sustainability_score'])}/5"
             if pd.notna(r["sustainability_score"]) else "N/A"
             for _, r in results.iterrows()])

    cmp_row("🔧 Co-op programs",
            [f"{int(r['programs_coop_count'])}"
             if pd.notna(r.get("programs_coop_count")) else "N/A"
             for _, r in results.iterrows()])

    print("─"*58)


In [33]:
#Run the display 

if results.empty:
    print("No recommendations could be generated. "
          "Try adjusting your academic average or category.")
else:
    weights = build_weights(student)

    print("\n" + "="*58)
    print("  🎓  YOUR UNIVERSITY RECOMMENDATIONS")
    print(f"  Student type : {student['student_status']}")
    print(f"  Area         : {student['category']}")
    print(f"  Your average : {student['avg']:.1f}%")
    print("="*58)

    for rank, (_, row) in enumerate(results.iterrows()):
        print_card(rank, row, student, weights)

    print_comparison(results, student)

    print(f"""
  💡  Next Steps
  ─────────────────────────────────────────────────
  • Visit each university's official website to
    explore specific program requirements
  • Check OUAC (ouac.on.ca) to start your application
  • Use the 'what-if' tool in Section 6 to adjust
    your priorities and see how rankings change
    """)


  🎓  YOUR UNIVERSITY RECOMMENDATIONS
  Student type : International
  Area         : Engineering & Technology
  Your average : 85.0%

  🥇  MATCH #1
  🎓  University of Ottawa
      Mathematics  |  Engineering & Technology

  📋  Key Facts
  ─────────────────────────────────────────────────
  💵  Annual cost (international):  $41,599/yr
  📊  Admission average:           89.0%  →  ❌ Reach
  💼  6-month employment rate:     100.0%
  🌍  QS World Rank:               219

  ✨  Why this matched you
  ─────────────────────────────────────────────────
  This match is driven by strong post-graduation employment rates and affordable tuition and cost of living.

  📈  Your priority scores
  ─────────────────────────────────────────────────
    💼 Employment         ██████████  100%
    💰 Cost               ██████░░░░  63%
    🌱 Sustainability     ███░░░░░░░  33%
    🤝 Diversity          ██████████  100%

  🥈  MATCH #2
  🎓  Carleton University
      Physical Science  |  Engineering & Technology

  📋  Ke

## Section 6 — What-If Tool

Lets the student adjust their priority ratings one at a time and immediately see how the top-3 recommendations change. No re-running the full quiz — just tweak and see.

This demonstrates the transparency of the weighted scoring model: the student can observe exactly how each priority shift affects which universities surface to the top.

In [35]:
## Adjustable dimensions
# These are the 6 dimensions the student rated in the quiz.
# We present them as a numbered menu so the student can
# pick which one to adjust.

ADJUSTABLE = [
    ("imp_jobs", "💼  Employment after graduation"),
    ("imp_cost", "💰  Low cost (tuition + living)"),
    ("imp_rank", "🏅  School ranking / prestige"),
    ("imp_sust", "🌱  Sustainability"),
    ("imp_div",  "🤝  Diversity & equity"),
    ("imp_coop", "🔧  Co-op availability"),
]

#Before/after comparison 
# Shows which schools were in the top 3 before and after
# the adjustment, highlighting any position changes.
# This is the key transparency feature — the student can
# see exactly what moved and by how much.

def print_before_after(before_results, after_results, changed_label, old_val, new_val):
    """
    Prints a compact comparison of top-3 before vs after
    a priority adjustment.
    """
    print(f"\n  🔄  Adjustment: {changed_label}")
    print(f"      {old_val}/5  →  {new_val}/5\n")

    before_names = [f"{i+1}. {r['school_name'].replace('University', 'U.')}"
                    for i, (_, r) in enumerate(before_results.iterrows())]
    after_names  = [f"{i+1}. {r['school_name'].replace('University', 'U.')}"
                    for i, (_, r) in enumerate(after_results.iterrows())]

    col_w = 26
    print(f"  {'BEFORE':<{col_w}}  {'AFTER'}")
    print(f"  {'─'*col_w}  {'─'*col_w}")
    for i in range(max(len(before_names), len(after_names))):
        b = before_names[i] if i < len(before_names) else "—"
        a = after_names[i]  if i < len(after_names)  else "—"
        # Flag if this position changed
        changed = "  ◄ changed" if b != a else ""
        print(f"  {b:<{col_w}}  {a}{changed}")
    print()

#Current priorities summary 
# Prints the student's current importance ratings so they
# always know their starting point before adjusting.

def print_current_priorities(s):
    print("\n  Your current priority ratings:")
    print("  ─────────────────────────────────────────")
    for i, (key, label) in enumerate(ADJUSTABLE, 1):
        val  = s[key]
        bar  = score_bar(val / 5)   # reuse bar helper from Section 5
        print(f"  {i}. {label:<35} {bar}  {val}/5")
    print()


In [36]:
# What-if loop ─────────────────────────────────────────
# Runs an interactive loop where the student can:
#   1. Pick a dimension to adjust
#   2. Enter a new rating (1–5)
#   3. See the updated top-3 recommendations immediately
#   4. Either adjust again or exit
#
# The loop always works from the ORIGINAL student profile
# plus any accumulated adjustments — so the student can
# layer multiple changes and always compare to the baseline.

def run_whatif(latest, student, original_results):
    """
    Runs the what-if adjustment loop.

    Parameters:
        latest           — the scored program table from Section 2
        student          — the original student dict from Section 3
        original_results — the original top-3 from Section 4
    """
    # Work on a copy so the original student dict is preserved
    adjusted = student.copy()
    current_results = original_results.copy()

    print("\n" + "="*58)
    print("  🔧  WHAT-IF TOOL")
    print("="*58)
    print("  Adjust your priority ratings to see how your")
    print("  recommendations change in real time.")
    print("  Type 'done' at any menu to exit.\n")

    while True:

        # Show current ratings
        print_current_priorities(adjusted)

        # Ask which dimension to adjust
        print("  Which priority would you like to adjust?")
        for i, (key, label) in enumerate(ADJUSTABLE, 1):
            print(f"    {i}. {label}")
        print("    7. Reset all priorities to original values")

        choice = input("\n  → Enter a number (1–7) or 'done': ").strip().lower()

        if choice == "done":
            print("\n  ✅ Exiting what-if tool. Final recommendations above.")
            break

        if choice == "7":
            # Reset to original student profile
            adjusted = student.copy()
            current_results = original_results.copy()
            print("\n  ↩  All priorities reset to original values.\n")
            continue

        if not (choice.isdigit() and 1 <= int(choice) <= 6):
            print("  ⚠  Please enter a number between 1 and 7, or 'done'.")
            continue

        idx = int(choice) - 1
        imp_key, label = ADJUSTABLE[idx]
        old_val = adjusted[imp_key]

        # Ask for new rating
        print(f"\n  Adjusting: {label}")
        print(f"  Current value: {old_val}/5")
        new_val_raw = input("  → New rating (1–5): ").strip()

        if not (new_val_raw.isdigit() and 1 <= int(new_val_raw) <= 5):
            print("  ⚠  Please enter a number between 1 and 5.")
            continue

        new_val = int(new_val_raw)

        if new_val == old_val:
            print(f"  ℹ  Rating unchanged at {old_val}/5.")
            continue

        # Apply the adjustment
        adjusted[imp_key] = new_val

        # Re-score with the adjusted profile
        new_results = score_candidates(latest, adjusted, top_n=3)

        if new_results.empty:
            print("  ⚠  No results with this adjustment. Try a different value.")
            adjusted[imp_key] = old_val   # revert
            continue

        # Show before/after
        print_before_after(current_results, new_results, label, old_val, new_val)

        # Show the new full cards
        weights = build_weights(adjusted)
        for rank, (_, row) in enumerate(new_results.iterrows()):
            print_card(rank, row, adjusted, weights)

        # Show comparison table
        print_comparison(new_results, adjusted)

        # Update current results for next iteration's "before"
        current_results = new_results


In [37]:
#Run the what-if tool
# Only runs if Section 4 produced results — no point in
# running the what-if tool if the initial scoring failed.

if not results.empty:
    run_whatif(latest, student, results)
else:
    print("⚠  What-if tool requires valid initial recommendations.")
    print("   Please re-run Sections 3 and 4 first.")


  🔧  WHAT-IF TOOL
  Adjust your priority ratings to see how your
  recommendations change in real time.
  Type 'done' at any menu to exit.


  Your current priority ratings:
  ─────────────────────────────────────────
  1. 💼  Employment after graduation      ██████████  5/5
  2. 💰  Low cost (tuition + living)      ██████████  5/5
  3. 🏅  School ranking / prestige        ████░░░░░░  2/5
  4. 🌱  Sustainability                   ██████░░░░  3/5
  5. 🤝  Diversity & equity               ██████████  5/5
  6. 🔧  Co-op availability               ██░░░░░░░░  1/5

  Which priority would you like to adjust?
    1. 💼  Employment after graduation
    2. 💰  Low cost (tuition + living)
    3. 🏅  School ranking / prestige
    4. 🌱  Sustainability
    5. 🤝  Diversity & equity
    6. 🔧  Co-op availability
    7. Reset all priorities to original values

  Adjusting: 🤝  Diversity & equity
  Current value: 5/5
   ℹ  21 program(s) removed by admission filter.

  🔄  Adjustment: 🤝  Diversity & equity
      5